In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
data_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [3]:
train_data = datasets.ImageFolder("DATASET/TRAIN", transform=data_transform)
test_data= datasets.ImageFolder("DATASET/TEST", transform=data_transform)

In [4]:
len(train_data)

22564

In [5]:
len(test_data)

2513

In [6]:
train_data[0][0].size()

torch.Size([3, 128, 128])

In [7]:
train_data_loader = DataLoader(train_data, shuffle=True, batch_size=128)
test_data_loader = DataLoader(test_data, shuffle=True, batch_size=128)

In [8]:
images, labels = next(iter(train_data_loader))

print("Batch shape:", images.shape)   # e.g. torch.Size([128, 3, 224, 224])
print("Labels shape:", labels.shape)  # e.g. torch.Size([128])
print("Labels:", labels)

Batch shape: torch.Size([128, 3, 128, 128])
Labels shape: torch.Size([128])
Labels: tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0,
        0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0,
        1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0,
        1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1])


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [10]:
from torch.nn.modules.conv import Conv2d

In [11]:
class ConvNeuralNetwork(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.linearForward = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )
    

    def forward(self, x):
        X = self.features(x)
        res = self.linearForward(X)
        return res

In [12]:
model = ConvNeuralNetwork()

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
for epoch in range(10):

  total_loss = 0
  for img, label in train_data_loader:
    img, label = img.to(device), label.to(device)

    optimizer.zero_grad()

    output = model(img)
    loss = criterion(output, label)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  avg_loss = total_loss / len(train_data_loader)
  print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

In [ ]:
model.eval()
images, labels = next(iter(test_data_loader))

with torch.no_grad():
    outputs = model(images.to(device))
    probs   = torch.softmax(outputs, dim=1)
    confs, preds = torch.max(probs, dim=1)
    print(confs, preds, labels)